In [0]:
-- Ingestão bronze
--
-- A bronze preserva o dado como veio. Nada de limpeza aqui: se der problema
-- depois, a gente precisa poder voltar na origem e comparar.
--
-- O poder do read_files: o MESMO comando lê CSV, planilha Excel, JSON e XML.
-- Ninguém converte nada antes. A Vértice tem projetos em três ferramentas:
--   · Excel       20 planilhas, uma por projeto, cada gerente do seu jeito
--   · ClickUp     25 exports da API em JSON (tarefas dentro de uma lista)
--   · MS Project  15 arquivos XML (uma <Task> por tarefa, com fases e baseline)
--
-- Duas coisas que valem explicar:
--   inferColumnTypes => false  ->  TUDO entra como texto (CSV e Excel). Se o
--                                  Databricks adivinhasse o tipo, ele erraria as
--                                  datas que vêm em dois formatos e o valor com
--                                  vírgula decimal. A sujeira sumiria antes de a
--                                  gente ver. No JSON e no XML deixamos inferir:
--                                  é o que mantém a estrutura (lista dentro de lista).
--   EXCEPT (_rescued_data)     ->  o read_files cria sozinho a coluna _rescued_data,
--                                  a "rede de segurança": o que não couber nas colunas
--                                  (uma coluna a mais numa linha, por exemplo) cai ali
--                                  em vez de sumir. Nos nossos arquivos ela vem vazia,
--                                  então a gente tira da tabela.

-- São 9 tabelas, uma pergunta do dono por trás de cada uma:
--   projetos, status_semanal                          o cadastro do PMO e o farol de toda sexta
--   excel_tarefas, clickup_tarefas, msproject_tarefas as 3 ferramentas (pergunta 1: quem atrasa)
--   custos                                            o financeiro (pergunta 2: quanto custa)
--   colaboradores, apontamentos                       RH e timesheet (pergunta 3: gente parada)
--   entregas_fornecedor                               compras (a consultoria que atrasa)
--
-- Antes de rodar, os arquivos precisam estar no volume (passos 3 e 4 do README):
--   /Volumes/lakehouse/bronze/raw/<sistema>/...
--
-- Rode com:
--   python3 scripts/run_sql.py aulas/aula-01-analista-sql/01-ingestao-bronze.sql --profile SEU-PERFIL

-- ---------- pmo ----------
-- projetos: o cadastro do PMO: cliente, valor, prazo e EM QUAL FERRAMENTA o projeto vive
CREATE OR REPLACE TABLE lakehouse.bronze.projetos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse/bronze/raw/pmo/projetos.csv',
                format => 'csv', header => true, inferColumnTypes => false);
-- status_semanal: toda sexta o gerente publica avanço, farol e bloqueios (texto livre)
CREATE OR REPLACE TABLE lakehouse.bronze.status_semanal AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse/bronze/raw/pmo/status_semanal.csv',
                format => 'csv', header => true, inferColumnTypes => false);

-- ---------- as três ferramentas de projeto ----------
-- excel_tarefas: as 20 planilhas de uma vez (é só apontar para a pasta).
-- A planilha tem cabeçalho escrito por gente ("% Concluído", "Início previsto"),
-- e tabela não aceita espaço no nome da coluna: renomeamos, sem mexer no valor.
CREATE OR REPLACE TABLE lakehouse.bronze.excel_tarefas AS
SELECT `ID`               AS id,
       `Fase`             AS fase,
       `Tarefa`           AS tarefa,
       `Responsável`      AS responsavel,
       `Início previsto`  AS inicio_previsto,
       `Término previsto` AS termino_previsto,
       `Início real`      AS inicio_real,
       `Término real`     AS termino_real,
       `% Concluído`      AS pct_concluido,
       `Horas previstas`  AS horas_previstas,
       url_decode(_metadata.file_name) AS arquivo
FROM read_files('/Volumes/lakehouse/bronze/raw/excel/',
                format => 'excel', headerRows => 1, inferColumnTypes => false);

-- clickup_tarefas: cada arquivo é UMA lista (um projeto) com as tarefas dentro.
-- explode() abre a lista: uma linha por tarefa. O resto fica como veio:
-- status é uma estrutura, datas são milissegundos, campos personalizados são uma lista.
CREATE OR REPLACE TABLE lakehouse.bronze.clickup_tarefas AS
SELECT list.id   AS list_id,
       list.name AS list_name,
       t.*,
       url_decode(_metadata.file_name) AS arquivo
FROM (
    SELECT list, explode(tasks) AS t, _metadata
    FROM read_files('/Volumes/lakehouse/bronze/raw/clickup/',
                    format => 'json', multiLine => true)
);

-- msproject_tarefas: cada <Task> do XML vira uma linha. Inclui as tarefas-resumo
-- (as fases e o projeto inteiro), que somam as horas das tarefas de baixo.
CREATE OR REPLACE TABLE lakehouse.bronze.msproject_tarefas AS
SELECT * EXCEPT (_rescued_data),
       url_decode(_metadata.file_name) AS arquivo
FROM read_files('/Volumes/lakehouse/bronze/raw/msproject/',
                format => 'xml', rowTag => 'Task');

-- ---------- rh e timesheet ----------
-- colaboradores: as 520 pessoas: papel, senioridade, custo/hora, admissão e desligamento
CREATE OR REPLACE TABLE lakehouse.bronze.colaboradores AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse/bronze/raw/rh/colaboradores.csv',
                format => 'csv', header => true, inferColumnTypes => false);
-- apontamentos: horas disponíveis x horas apontadas, por pessoa, projeto e semana
CREATE OR REPLACE TABLE lakehouse.bronze.apontamentos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse/bronze/raw/timesheet/apontamentos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

-- ---------- financeiro ----------
-- custos: cada lançamento de custo real (folha alocada, consultoria, cloud, licenças...)
CREATE OR REPLACE TABLE lakehouse.bronze.custos AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse/bronze/raw/financeiro/custos.csv',
                format => 'csv', header => true, inferColumnTypes => false);

-- ---------- compras ----------
-- entregas_fornecedor: o que cada consultoria prometeu entregar, e quando entregou
CREATE OR REPLACE TABLE lakehouse.bronze.entregas_fornecedor AS
SELECT * EXCEPT (_rescued_data)
FROM read_files('/Volumes/lakehouse/bronze/raw/compras/entregas_fornecedor.csv',
                format => 'csv', header => true, inferColumnTypes => false);

-- ---------- conferência ----------
-- 9 tabelas. Se alguma vier com 0 linhas, o arquivo não subiu para o volume.
SELECT 'projetos' AS tabela, COUNT(*) AS linhas FROM lakehouse.bronze.projetos
UNION ALL SELECT 'status_semanal', COUNT(*) FROM lakehouse.bronze.status_semanal
UNION ALL SELECT 'excel_tarefas', COUNT(*) FROM lakehouse.bronze.excel_tarefas
UNION ALL SELECT 'clickup_tarefas', COUNT(*) FROM lakehouse.bronze.clickup_tarefas
UNION ALL SELECT 'msproject_tarefas', COUNT(*) FROM lakehouse.bronze.msproject_tarefas
UNION ALL SELECT 'colaboradores', COUNT(*) FROM lakehouse.bronze.colaboradores
UNION ALL SELECT 'apontamentos', COUNT(*) FROM lakehouse.bronze.apontamentos
UNION ALL SELECT 'custos', COUNT(*) FROM lakehouse.bronze.custos
UNION ALL SELECT 'entregas_fornecedor', COUNT(*) FROM lakehouse.bronze.entregas_fornecedor
ORDER BY linhas DESC;

In [0]:
CREATE OR REPLACE VIEW lakehouse.bronze.rascunho_tarefas
COMMENT 'Rascunho da noite 1: tarefas do Excel, ClickUp e MS Project empilhadas à mão. Na noite 2 vira silver.tarefas.'
AS

-- ---------- EXCEL ----------
SELECT -- o projeto está no nome do arquivo: 'Cronograma PRJ-002.xlsx' → pega "PRJ-" + 3 dígitos
       regexp_extract(arquivo, 'PRJ-[0-9]{3}', 0)                         AS projeto_id,
       'excel'                                                            AS ferramenta,
       tarefa,
       -- data com barra ('06/03/2025') → lê como dd/MM/yyyy;
       -- senão é o número de série do Excel (45700 = dias contados desde 30/12/1899)
       CASE WHEN inicio_previsto LIKE '%/%' THEN to_date(inicio_previsto, 'dd/MM/yyyy')
            ELSE date_add(DATE'1899-12-30', CAST(inicio_previsto AS INT)) END   AS inicio_baseline,
       CASE WHEN termino_previsto LIKE '%/%' THEN to_date(termino_previsto, 'dd/MM/yyyy')
            ELSE date_add(DATE'1899-12-30', CAST(termino_previsto AS INT)) END  AS fim_baseline,
       -- '45%' ou '64,4%' → 45 e 64.4;  0.45 → 45;  e LEAST corta quem passou de 100
       LEAST(100, CASE WHEN pct_concluido LIKE '%\%%'
                       THEN CAST(replace(replace(pct_concluido, '%', ''), ',', '.') AS DOUBLE)
                       ELSE CAST(pct_concluido AS DOUBLE) * 100 END)      AS pct_concluido,
       CAST(replace(horas_previstas, 'h', '') AS DOUBLE)                  AS horas
FROM lakehouse.bronze.excel_tarefas
WHERE tarefa <> 'TOTAL'                     -- fora a linha de TOTAL (exemplo 03)

UNION ALL
-- ---------- CLICKUP ----------
-- O ClickUp não sabe o que é PRJ-xxx. O cadastro do PMO sabe: guarda o código da lista do ClickUp.
-- O JOIN junta as duas tabelas (c = clickup, p = projetos); o OR casa pelo nome quando falta o código.
-- O % e as datas da baseline estão escondidos numa lista de campos personalizados: o filter()
-- procura o campo pelo nome. As datas e as horas vêm em milissegundos.
SELECT p.projeto_id,
       'clickup',
       c.name,
       to_date(timestamp_millis(CAST(filter(c.custom_fields, f -> f.name = 'Início baseline')[0].value AS BIGINT))),
       to_date(timestamp_millis(CAST(filter(c.custom_fields, f -> f.name = 'Fim baseline')[0].value AS BIGINT))),
       CAST(filter(c.custom_fields, f -> f.name = 'Progresso (%)')[0].value AS DOUBLE),
       c.time_estimate / 3600000                                           -- milissegundos → horas
FROM lakehouse.bronze.clickup_tarefas c
JOIN lakehouse.bronze.projetos p
  ON p.codigo_ferramenta = c.list_id OR p.nome_projeto = c.list_name

UNION ALL
-- ---------- MS PROJECT ----------
-- As datas da baseline estão dentro da "caixinha" Baseline; as horas vêm como 'PT40H0M0S' (→ 40).
-- Summary = 0: só as tarefas de verdade. A tarefa-resumo (a fase) soma as de baixo e contaria duas vezes.
SELECT regexp_extract(arquivo, 'PRJ-[0-9]{3}', 0),
       'msproject',
       Name,
       to_date(Baseline.Start),
       to_date(Baseline.Finish),
       CAST(PercentComplete AS DOUBLE),
       CAST(regexp_extract(Work, 'PT([0-9]+)H', 1) AS DOUBLE)
FROM lakehouse.bronze.msproject_tarefas
WHERE Summary = 0;


In [0]:
-- ============================================
-- EXEMPLO 06: QUANTO CUSTA? O estouro e o farol do gerente
-- ============================================
-- Conceito: juntar várias consultas com WITH e fazer conta de KPI
-- Pergunta de negócio: SE NADA MUDAR, QUANTO A VÉRTICE VAI PERDER? E O FAROL MOSTRA ISSO?
-- Usa o rascunho do exemplo 05 (bronze.rascunho_tarefas): rode o 05 antes.
--
-- As contas (todas sobre o orçamento de custo do projeto):
--   valor agregado (EV) = orçamento × fração pronta          (quanto vale o que já foi entregue)
--   CPI = EV ÷ custo real                                    abaixo de 1 é estouro: cada R$ 1
--                                                            gasto entregou menos de R$ 1
--   EAC = orçamento ÷ CPI                                    quanto o projeto vai custar no fim,
--                                                            se o ritmo continuar
--   estouro previsto = EAC − orçamento
--
-- A consulta tem 3 pedaços (WITH), cada um uma pergunta simples:
--   valor   quanto de cada projeto está pronto, e quanto devia estar (o exemplo 05)
--   custo   quanto cada projeto já gastou (o exemplo 04, por projeto)
--   farol   a cor que o gerente publicou nesta sexta, 16/10

-- 1. Projeto a projeto: prazo, custo, estouro, e o farol do gerente.
--    Deve aparecer: 41 projetos, do pior para o melhor. Repare na coluna farol_do_gerente:
--    PRJ-012, 028, 034 e 042 estão VERDES com SPI abaixo de 0,9. São os "melancias":
--    verdes por fora, vermelhos por dentro.
WITH valor AS (
    SELECT projeto_id,
           SUM(horas)                       AS horas_totais,
           SUM(horas * pct_concluido / 100) AS horas_agregadas,
           SUM(horas * GREATEST(0, LEAST(1,
               datediff(DATE'2026-10-16', inicio_baseline) / datediff(fim_baseline, inicio_baseline)
           )))                               AS horas_planejadas
    FROM lakehouse.bronze.rascunho_tarefas
    GROUP BY projeto_id
),
custo AS (
    SELECT projeto_id,
           SUM(CAST(replace(valor, ',', '.') AS DECIMAL(18,2))) AS custo_real
    FROM lakehouse.bronze.custos
    GROUP BY projeto_id
),
farol AS (
    SELECT projeto_id, lower(farol) AS farol
    FROM lakehouse.bronze.status_semanal
    WHERE data_status = '2026-10-16'
)
SELECT p.projeto_id,
       p.cliente,
       ROUND(v.horas_agregadas / v.horas_planejadas, 2)                          AS spi,
       ROUND(CAST(p.orcamento_custo AS DECIMAL(18,2))
             * v.horas_agregadas / v.horas_totais / c.custo_real, 2)              AS cpi,
       f.farol                                                                   AS farol_do_gerente,
       -- estouro = EAC − orçamento = orçamento ÷ CPI − orçamento
       ROUND((c.custo_real * v.horas_totais / v.horas_agregadas
              - CAST(p.orcamento_custo AS DECIMAL(18,2))) / 1000000, 2)           AS estouro_milhoes
FROM lakehouse.bronze.projetos p
JOIN valor v USING (projeto_id)
JOIN custo c USING (projeto_id)
JOIN farol f USING (projeto_id)
WHERE p.status = 'em_andamento'
ORDER BY spi;

In [0]:
-- 2. O número para o dono: quanto a carteira vai estourar, somando só quem estoura.
--    (Projeto que está gastando menos que o previsto não "compensa" o que outro queima.)
--    Nada novo: as mesmas contas do bloco 1, agora somadas. GREATEST(0, x) zera quem não estoura.
--    Deve aparecer: R$ 6,47 mi. A resposta da pergunta 2.
WITH valor AS (
    SELECT projeto_id,
           SUM(horas)                       AS horas_totais,
           SUM(horas * pct_concluido / 100) AS horas_agregadas
    FROM lakehouse.bronze.rascunho_tarefas
    GROUP BY projeto_id
),
custo AS (
    SELECT projeto_id,
           SUM(CAST(replace(valor, ',', '.') AS DECIMAL(18,2))) AS custo_real
    FROM lakehouse.bronze.custos
    GROUP BY projeto_id
)
SELECT ROUND(SUM(GREATEST(0,
           c.custo_real * v.horas_totais / v.horas_agregadas
           - CAST(p.orcamento_custo AS DECIMAL(18,2)))) / 1000000, 2) AS estouro_previsto_milhoes
FROM lakehouse.bronze.projetos p
JOIN valor v USING (projeto_id)
JOIN custo c USING (projeto_id)
WHERE p.status = 'em_andamento';